# Table 6.1 (Section 6.3)

Monte Carlo coverage study for the Elephant Random Walk. This notebook
produces the numerical results, the LaTeX source for Table 6.1, and the
diffusive-coverage plot.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
def simulate_erw_coverage(
    p_values=(0.25, 0.50, 0.60, 0.65, 0.75, 0.80, 0.90),
    n_values=(500, 1000, 10000, 50000, 250000),
    N=5000,
    q=0.75,
    alpha=0.05,
    seed=20260525,
):
    """
    Monte Carlo coverage study for the Elephant Random Walk.

    Parameters
    ----------
    p_values : tuple
        True memory parameters.
    n_values : tuple
        Path lengths.
    N : int
        Number of simulated paths per p-value.
    q : float
        Initial bias: P(X_1 = 1) = q.
    alpha : float
        Significance level. For a 95% CI, use alpha=0.05.
    seed : int or None
        Random seed for reproducibility.

    Returns
    -------
    df : pandas.DataFrame
        Table with mean estimator, diffusive coverage, valid rate,
        interval lengths, and general coverage.
    """

    rng = np.random.default_rng(seed)

    p_values = np.array(p_values, dtype=float)
    n_values = sorted(n_values)
    n_set = set(n_values)
    max_n = max(n_values)

    # Standard normal 0.975 quantile.
    z = 1.959963984540054

    # ERW parameter alpha_ERW = 2p - 1.
    a = 2 * p_values - 1
    m = len(p_values)

    # First increment: P(X_1 = 1) = q.
    X1 = np.where(rng.random((m, N)) < q, 1, -1).astype(np.int8)

    # Current position S_k.
    S = X1.astype(np.int32)

    # Estimator components:
    # numerator   = sum_{k=1}^{n-1} (S_k/k)(X_{k+1} + S_k/k)
    # denominator = sum_{k=1}^{n-1} (S_k/k)^2
    numerator = np.zeros((m, N), dtype=float)
    denominator = np.zeros((m, N), dtype=float)

    rows = []

    for k in range(1, max_n):
        # At this point, S is S_k.
        S_over_k = S / k

        # Add denominator term before generating X_{k+1}.
        denominator += S_over_k**2

        # Equivalent ERW simulation formula:
        # P(X_{k+1}=1 | F_k) = 1/2 * (1 + (2p-1) S_k/k).
        prob_plus = 0.5 * (1 + a[:, None] * S_over_k)
        np.clip(prob_plus, 0.0, 1.0, out=prob_plus)

        X_next = np.where(rng.random((m, N)) < prob_plus, 1, -1).astype(np.int8)

        # Add numerator term.
        numerator += S_over_k * (X_next + S_over_k)

        # Update position.
        S += X_next.astype(np.int32)

        n = k + 1

        if n in n_set:
            phat = numerator / (2 * denominator)
            Vn = 4 * denominator

            # General exact confidence interval:
            # J_n = [phat - h_general, phat + h_general].
            h_general = 2 * np.sqrt(3 * n * np.log(2 / alpha)) / Vn
            length_general = 2 * h_general

            cover_general = (
                (phat - h_general <= p_values[:, None])
                & (p_values[:, None] <= phat + h_general)
            ).mean(axis=1)

            for i, p in enumerate(p_values):
                if p < 0.75:
                    # Diffusive CI is only defined when phat < 3/4.
                    valid = (3 - 4 * phat[i]) >= 0
                    valid_rate = valid.mean()

                    h_diff = np.full(N, np.nan)
                    h_diff[valid] = (
                        z * np.sqrt(3 - 4 * phat[i, valid])
                        / (2 * np.sqrt(np.log(n)))
                    )

                    # Undefined diffusive intervals are counted as non-coverage.
                    cover_diff = np.mean(
                        valid
                        & (phat[i] - h_diff <= p)
                        & (p <= phat[i] + h_diff)
                    )

                    # Mean length only over defined diffusive intervals.
                    length_diff = np.nanmean(2 * h_diff)
                else:
                    valid_rate = np.nan
                    cover_diff = np.nan
                    length_diff = np.nan

                rows.append({
                    "p": p,
                    "n": n,
                    "mean_phat": phat[i].mean(),
                    "diff_valid": valid_rate,
                    "coverage_diffusive": cover_diff,
                    "mean_length_diffusive": length_diff,
                    "coverage_general": cover_general[i],
                    "mean_length_general": length_general[i].mean(),
                })

    df = pd.DataFrame(rows)
    df = df.sort_values(["p", "n"]).reset_index(drop=True)

    return df

In [ ]:
def make_latex_table(df):
    """
    Convert the simulation dataframe into a LaTeX table.
    Requires \usepackage{booktabs}.
    """

    lines = []

    lines.append(r"\begin{table}[htbp]")
    lines.append(r"\centering")
    lines.append(r"\scriptsize")
    lines.append(r"\begin{tabular}{cccccccc}")
    lines.append(r"\toprule")
    lines.append(
        r"$p$ & $n$ & Mean $\widehat p_n$ & Diff. valid & Diff. cov. "
        r"& Mean diff. length & Gen. cov. & Mean gen. length \\"
    )
    lines.append(r"\midrule")

    p_values = df["p"].unique()

    for j, p in enumerate(p_values):
        sub = df[df["p"] == p]

        for _, row in sub.iterrows():
            p_str = f"{row['p']:.2f}"
            n_str = f"{int(row['n'])}"

            mean_phat = f"{row['mean_phat']:.4f}"

            if pd.isna(row["diff_valid"]):
                diff_valid = "--"
                diff_cov = "--"
                diff_len = "--"
            else:
                diff_valid = f"{row['diff_valid']:.4f}"
                diff_cov = f"{row['coverage_diffusive']:.4f}"
                diff_len = f"{row['mean_length_diffusive']:.4f}"

            gen_cov = f"{row['coverage_general']:.4f}"
            gen_len = f"{row['mean_length_general']:.4f}"

            lines.append(
                f"{p_str} & {n_str} & {mean_phat} & {diff_valid} & "
                f"{diff_cov} & {diff_len} & {gen_cov} & {gen_len} \\\\"
            )

        if j < len(p_values) - 1:
            lines.append(r"\midrule")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(
        r"\caption{Monte Carlo coverage study for the ERW with $q=0.75$, "
        r"$N=5000$ simulated paths, seed $20260525$, and confidence level "
        r"$0.95$. The column ``Mean $\widehat p_n$'' gives the empirical "
        r"average of the estimator. The column ``Diff. valid'' gives the "
        r"proportion of simulations for which the diffusive confidence "
        r"interval is defined, i.e. $\widehat p_n<3/4$. The diffusive "
        r"confidence interval and its length are only reported for "
        r"$p<3/4$. The reported interval lengths are not truncated to the "
        r"parameter space $[0,1]$.}"
    )
    lines.append(r"\label{tab:mc-coverage-study}")
    lines.append(r"\end{table}")

    return "\n".join(lines)

In [ ]:
def plot_diffusive_coverage(df, p_values=(0.25, 0.50, 0.60, 0.65)):
    """
    Plot empirical coverage of the diffusive CI for p < 3/4.
    """

    fig, axes = plt.subplots(2, 2, figsize=(12, 5.5), sharex=False)
    axes = axes.ravel()

    for ax, p in zip(axes, p_values):
        sub = df[df["p"] == p]

        x = sub["n"].to_numpy()
        y = sub["coverage_diffusive"].to_numpy()

        ax.vlines(x, ymin=min(0.84, np.nanmin(y) - 0.02), ymax=y, alpha=0.8)
        ax.scatter(x, y, zorder=3)

        ax.axhline(0.95, linestyle="--", linewidth=1)

        ax.set_xscale("log")
        ax.set_title(rf"$p={p:.2f}$")
        ax.set_xlabel(r"$n$")
        ax.set_ylabel(r"$\widehat{\mathrm{Cov}}^{\mathrm{diff}}_n$")

        ax.set_xticks(x)
        ax.set_xticklabels([f"{int(v):,}" for v in x])

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.tight_layout()
    plt.show()

In [ ]:
# Run simulation.
df = simulate_erw_coverage()

# Print numerical dataframe.
print(df.to_string(index=False))

# Print LaTeX table.
latex_code = make_latex_table(df)
print(latex_code)

# Optional: plot diffusive coverage.
plot_diffusive_coverage(df)